In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
def main():
    # Step 1: Data Loading & Preprocessing
    # Load the NBER CES dataset
    file_path = r"C:\Users\ofek3\Source\02_Idea_1973_Code\01_Sourse_Idea\data\nberces5818v1_n2012.csv"
    df = pd.read_csv(file_path)

    # Create real variables by deflating energy and value added
    df['real_energy'] = df['energy'] / df['pien']
    df['real_vadd'] = df['vadd'] / df['piship']

    # Create logarithmic variables (including capital)
    df['log_vadd'] = np.log(df['real_vadd'])
    df['log_emp'] = np.log(df['emp'])
    df['log_cap'] = np.where(df['cap'] > 0, np.log(df['cap']), np.nan)

    # Calculate real energy intensity
    df['real_energy_intensity'] = df['real_energy'] / df['real_vadd']

    # Sort data to accurately calculate value added growth rate year over year
    df = df.sort_values(['naics', 'year'])
    df['vadd_growth'] = df.groupby('naics')['real_vadd'].pct_change() * 100

    # Filter the dataset to include only years up to 1999
    df = df[df['year'] <= 1999].copy()

    # Step 2: Treatment Assignment based on Pre-Crisis Average (1968-1972)
    # Isolate the pre-crisis baseline
    df_base = df[(df['year'] >= 1968) & (df['year'] <= 1972)].copy()
    
    # Calculate the average real energy intensity across all industries in the base period
    mean_intensity = df_base.groupby('naics')['real_energy_intensity'].mean()

    # Calculate terciles (33% and 66%)
    energy_thresholds = mean_intensity.quantile([0.33, 0.66])

    # Function to classify into terciles
    def classify_energy(val):
        if val <= energy_thresholds[0.33]: return 'Low Energy'
        elif val <= energy_thresholds[0.66]: return 'Medium Energy'
        else: return 'High Energy'

    # Map the classification to all years for each industry
    energy_map = mean_intensity.apply(classify_energy)
    df['energy_group'] = df['naics'].map(energy_map)

    # Create a dummy variable for high energy intensity
    df['is_high_energy'] = (df['energy_group'] == 'High Energy').astype(int)

    # Drop any industries that did not exist in the baseline period
    df = df.dropna(subset=['energy_group']).copy()

    # Step 3: Industry Size Classification (Face Validity)
    # Classify industries by average employees before the crisis (up to 1972)
    industry_size = df[df['year'] <= 1972].groupby('naics')['emp'].mean()
    size_thresholds = industry_size.quantile([0.33, 0.66])

    def classify_size(emp_avg):
        if pd.isna(emp_avg): return np.nan
        if emp_avg <= size_thresholds[0.33]: return 'Small Industry'
        elif emp_avg <= size_thresholds[0.66]: return 'Medium Industry'
        else: return 'Large Industry'

    df['industry_size_cat'] = df['naics'].map(industry_size.apply(classify_size))

    # Step 4: Define Time Periods (Including 1973 as the baseline/end of pre-crisis)
    def assign_period(year):
        if 1958 <= year <= 1973:
            return "1. Pre-Crisis (1958-1973)"
        elif 1974 <= year <= 1999:
            return "2. Post-Crisis (1974-1999)"
        else:
            return np.nan 

    # Apply the function to create the new period column
    df['period'] = df['year'].apply(assign_period)
    df = df.dropna(subset=['period']).copy()

    # Create a subset of only High and Low energy for extreme group comparison
    df_extremes = df[df['energy_group'].isin(['High Energy', 'Low Energy'])].copy()

    # Step 5: Descriptive Statistics
    continuous_vars = ['tfp5', 'real_energy_intensity', 'log_vadd', 'log_emp', 'log_cap', 'vadd_growth']
    stat_funcs = ['mean', 'std', 'min', 'max']

    # Overall descriptive stats (Full Sample) - rounded to 3 decimal places
    desc_overall = df.groupby('period')[continuous_vars].agg(stat_funcs).round(3)
    desc_overall.to_csv('Descriptive_Stats_Overall.csv')

    # Grouped descriptive stats (Comparing extremes: High vs Low Energy)
    desc_grouped = df_extremes.groupby(['period', 'energy_group'])[continuous_vars].agg(stat_funcs)
    
    # Calculate counts of observations and unique industries for each group
    group_counts = df_extremes.groupby(['period', 'energy_group']).agg(
        Total_Observations=('naics', 'count'),
        Unique_Industries=('naics', 'nunique')
    )
    
    # Add MultiIndex columns to group_counts so it merges cleanly with desc_grouped
    group_counts.columns = pd.MultiIndex.from_product([['Sample_Info'], group_counts.columns])
    
    # Merge the counts with the descriptive statistics and round to 3 decimal places
    desc_grouped = pd.concat([group_counts, desc_grouped], axis=1).round(3)
    
    # Export grouped stats to CSV
    desc_grouped.to_csv('Descriptive_Stats_Grouped.csv')
    
    # Print the sample info to the console for quick verification
    print("--- Observations and Unique Industries per Group ---")
    print(desc_grouped['Sample_Info'])
    print("-" * 50)

    # Export and print Industry Size Distribution Cross-tabulation
    size_dist = pd.crosstab(
        df_extremes.drop_duplicates('naics')['energy_group'], 
        df_extremes.drop_duplicates('naics')['industry_size_cat']
    )
    size_dist.to_csv('Industry_Size_Distribution.csv')
    print("--- Industry Size Distribution (Face Validity) ---")
    print(size_dist)
    print("-" * 50)

    # Step 6: Academic Visualizations (EDA)
    sns.set_style("whitegrid")

    # Visualization A: Boxplots (Comparing High vs Low Energy Extremes)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    sns.boxplot(data=df_extremes, x='period', y='tfp5', hue='energy_group', palette={'High Energy': 'red', 'Low Energy': 'green'}, ax=axes[0])
    axes[0].set_title('TFP5 by Period and Energy Group')

    sns.boxplot(data=df_extremes, x='period', y='real_energy_intensity', hue='energy_group', palette={'High Energy': 'red', 'Low Energy': 'green'}, ax=axes[1])
    axes[1].set_title('Real Energy Intensity by Period')
    # Limit y-axis slightly to account for extreme positive skew/outliers
    upper_limit = df_extremes['real_energy_intensity'].quantile(0.95)
    axes[1].set_ylim(0, upper_limit) 

    # Plotting Capital instead of VADD to show resource allocation changes
    sns.boxplot(data=df_extremes, x='period', y='log_cap', hue='energy_group', palette={'High Energy': 'red', 'Low Energy': 'green'}, ax=axes[2])
    axes[2].set_title('Log Capital by Period and Energy Group')

    plt.tight_layout()
    plt.savefig('Boxplots_Extremes.png', dpi=300)
    plt.close()

    # Visualization B: Industry Size Distribution Bar Chart
    plt.figure(figsize=(10, 6))
    size_dist_plot = size_dist.reindex(columns=['Small Industry', 'Medium Industry', 'Large Industry'])
    
    # Plotting the cross-tabulation
    ax = size_dist_plot.T.plot(kind='bar', color=['red', 'green'], alpha=0.8, figsize=(10, 6))
    plt.title('Industry Size Distribution by Energy Group (Pre-Crisis)')
    plt.ylabel('Number of Industries')
    plt.xlabel('Industry Size Category')
    plt.xticks(rotation=0)
    plt.legend(title='Energy Group')
    
    plt.tight_layout()
    plt.savefig('Industry_Size_Distribution.png', dpi=300)
    plt.close()

    # Visualization C: Correlation Heatmap
    plt.figure(figsize=(10, 8))
    # Round the correlation matrix directly to 3 places
    corr_matrix = df_extremes[continuous_vars].corr().round(3)
    # Set the format to .3f to reflect the 3 decimal places
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".3f", vmin=-1, vmax=1)
    plt.title('Pearson Correlation Matrix (High & Low Energy)')
    
    plt.tight_layout()
    plt.savefig('Correlation_Heatmap.png', dpi=300)
    plt.close()

    print("Data processing complete. CSV files and PNG figures have been generated successfully.")

if __name__ == "__main__":
    main()

C:\Users\ofek3\AppData\Local\Temp\ipykernel_16260\3892571029.py:21: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['vadd_growth'] = df.groupby('naics')['real_vadd'].pct_change() * 100


--- Observations and Unique Industries per Group ---
                                         Total_Observations  Unique_Industries
period                     energy_group                                       
1. Pre-Crisis (1958-1973)  High Energy                 2016                126
                           Low Energy                  1904                119
2. Post-Crisis (1974-1999) High Energy                 3276                126
                           Low Energy                  3094                119
--------------------------------------------------
--- Industry Size Distribution (Face Validity) ---
industry_size_cat  Large Industry  Medium Industry  Small Industry
energy_group                                                      
High Energy                    39               35              49
Low Energy                     41               42              36
--------------------------------------------------
Data processing complete. CSV files and PNG figures 

<Figure size 1000x600 with 0 Axes>